In [ ]:
import numpy as np
import scipy.constants as constants

from helper_functions import (radial, fourier_shell_correlation, interpolated_intercepts, write_text)

import seaborn as sns
sns.set_theme()

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import h5py
     
import glob, os

e = constants.elementary_charge
h = constants.Planck
c = constants.speed_of_light

saveFig = False
extension = 'prot_wat' # 'prot_only' or 'prot_wat'
calcFSC = True
multiFSC = True

base_dir = 'average_reconstructions/'
mrc_dir = 'average_reconstructions/chimerax/'
    
npy_chimerax = sorted(glob.glob(mrc_dir + '*.npy',recursive = True))
npy_chimerax

<h2> Fourier Shell Correlation </h2> 
Assuming the two 3D Fourier volumes (FT of two 3D electron densities) contain an additive noise component, the FSC for this assumption leads to correct value for zero noise 
case, namely FSC=1, and when there is zero signal that the FSC is equal to the inverse of the number of voxels within each shell used to integrate. 
$$FSC(r_i)\: = \frac{SNR(r_i)+2/\sqrt{n(r_i)}\times\sqrt{SNR(r_i)}+1/\sqrt{n)r_i)}} {SNR(r_i)+2/\sqrt{n(r_i)}\times\sqrt{SNR(r_i)}+1}$$
From this, a threshold curve can now be defined:
$$T_{1.0}(r_i)\: = \frac{0.5+2.4142\times1/\sqrt{n(r_i)}}{1.5+1.4242\times1/\sqrt{n(r_i)}}$$
The previous is actually the 1-bit threshold, at which the reconstruction has reached an SNR of 1.0 for the entire reconstruction (0.5 per half data set). This threshold might be too strict, so the more common
curve is the 0.5-bit threshold, where each reconstruction reaches a SNR of 0.4242 for the entire reconstruction:
$$T_{0.5}(r_i)\: = \frac{0.2071+1.9102\times1/\sqrt{n(r_i)}}{1.2071+0.9102\times1/\sqrt{n(r_i)}}$$
Depending on the symmetry, it might be necessary to divide the $n(r_i)$ by the number of asymmetric units. 

In [ ]:
# Opening reference volume
name_ref = f'{mrc_dir}'+'1ss8_denss_rs.npy'
vol_ref = np.load(name_ref) # DENSS model

dim = vol_ref.shape[0]
center = dim//2

phot_eV = 9000
edge_pixel = dim - dim//2
write_text(f'Edge pixel: {edge_pixel}\n')
d_detector = 0.5
s_pixel = 800e-6 # for 6x downsampling

phot_m = (h * c) / (phot_eV * e)

theta_edge = 0.5 * np.arctan((edge_pixel*s_pixel)/d_detector) 
edge_res = phot_m/(2.0*np.sin(theta_edge))*1e9
edge_res_inv = 1 / edge_res

write_text(f'Edge resolution: {edge_res} nm\n')
write_text(f'Edge resolution: {edge_res_inv} nm^-1\n')

center_to_corner = np.sqrt((edge_pixel * s_pixel)**2+(edge_pixel * s_pixel)**2)
theta_max = 0.5 * np.arctan(center_to_corner/d_detector)
corner_res = phot_m / (2.0 * np.sin(theta_max)) * 1e9
corner_res_inv = 1 / corner_res

write_text(f'Corner resolution: {corner_res} nm\n')
write_text(f'Corner resolution: {corner_res_inv} nm^-1\n')

s_map_voxel = 2.3776362931527845e-10 # in m^-1
pix_fourier = 1 / (dim * s_map_voxel * 1e9) # in nm^-1
D_object = 15e-9 / s_map_voxel

In [ ]:
if calcFSC:
    for dens_c, dens_name in enumerate(npy_chimerax):
        dens_f = dens_name.split(sep='/')[2]
            
        if dens_f == '1ss8_denss_rs.npy':
            write_text('Reference volume encountered, no FSC calculation!\n')
        else:
            write_text(f'Calculating FSC with reference for: {dens_f}...\n')
            name_alg = f'{mrc_dir}{dens_f}'
            short_name = dens_name.split(sep='/')[2].split(sep='.')[0]
            vol_alg = np.load(name_alg) # super-reconstruction density - phased EMC model
        
            fsc, n_ri = fourier_shell_correlation(vol_alg,vol_ref)
            max_points = fsc.shape[0]
        
            # 0.5-bit curve
            n_ri = n_ri.sum(axis=(1,2,3))
            n_ri_inv = 1 / np.sqrt(n_ri)
        
            # corrected 0.5-bit curve
            corr_half = True
            c_half = 'no_halfcorr'
            if corr_half:
                n_ri = (n_ri / 2) * (1.5 * (D_object / dim))**2
                n_ri_inv = 1 / np.sqrt(n_ri)
                c_half = 'halfcorr'
                
            half_bit = (0.2071 + 1.9102 * n_ri_inv) / (1.2071 + 0.9102 * n_ri_inv)
        
            fsc_average = (n_ri*fsc).sum() / n_ri.sum()
            write_text(f'Average FSC: {fsc_average}\n')
        
            fp_resolution_fsc_inv = np.arange(0, max_points) * pix_fourier
        
            xci_half, yci_half = interpolated_intercepts(fp_resolution_fsc_inv, fsc, half_bit)
            
            plt.figure(dpi=140)
            plt.plot(fp_resolution_fsc_inv, fsc, c=mcolors.XKCD_COLORS['xkcd:jungle green'], linestyle='-')
            plt.plot(fp_resolution_fsc_inv, half_bit, c='k', linestyle='--')
            plt.xlim([0.0, fp_resolution_fsc_inv[-1]])
            plt.ylim([0, 1.01])
            plt.xlabel('|q| $(nm^{-1})$', weight='bold')
            plt.ylabel('FSC', weight='bold')
            plt.title(f'{short_name}')
            plt.legend([f'FSC','0.5-bit curve'],frameon=False, prop=dict(weight='bold', size=8), fontsize=0.5, loc=1)
            plt.axvline(x=fp_resolution_fsc_inv[-1], ymin=0, ymax=1, c=mcolors.XKCD_COLORS['xkcd:green'],linestyle='--', alpha=0.4); # edge-resolution of EMC reconstruction
        
            if xci_half.size != 0:
                if corr_half:
                    if xci_half.size == 1:
                        if 1/xci_half > 10: # don't consider first intersection FSC with threshold
                            write_text(f'Intersection(s) FSC with half-bit criterion: - nm\n')
                        else:
                            plt.plot(xci_half, yci_half, 'ko', ms=7)
                            write_text(f'Intersection(s) FSC with half-bit criterion: {1/xci_half} nm\n')
                    else:
                        if 1/xci_half[0] > 10: # don't consider first intersection FSC with threshold
                            plt.plot(xci_half[1:], yci_half[1:], 'ko', ms=7)
                            write_text(f'Intersection(s) FSC with half-bit criterion: {1/xci_half[1:]} nm\n')
                else:
                    plt.plot(xci_half, yci_half, 'ko', ms=7)
                    write_text(f'Intersection(s) FSC with half-bit criterion: {1/xci_half} nm\n')
                    
            if saveFig:
                save_name = short_name
                with h5py.File(f'figures_fsc/'+save_name+f'_{c_half}_FSC.h5', mode='a') as fsc_handle:
                    fsc_handle['fsc_r'] = fsc
                    fsc_handle['fp_res_inv_nm'] = fp_resolution_fsc_inv
                    if corr_half:
                        if xci_half.size == 1: # one intersection
                            if 1/xci_half < 10:
                                fsc_handle['res_r_nm'] = 1/xci_half
                        elif xci_half.size > 0: # more than one intersection
                            if 1/xci_half[0] > 10:
                                fsc_handle['res_r_nm'] = 1/xci_half[1:]
                        elif xci_half.size == 0: # no intersection
                            pass
                    else:
                        fsc_handle['res_r_nm'] = 1/xci_half
                        
                    fsc_handle['fsc_average'] = fsc_average
                    fsc_handle['n_ri'] = n_ri
                    fsc_handle['pdb_reference'] = name_ref
                    plt.savefig(f'figures_fsc/'+save_name+f'_{c_half}_FSC.pdf', dpi=150, bbox_inches='tight', pad_inches=0.0);
            write_text('\n')

<h2> Plotting multiple FSCs in single figure </h2> 

In [ ]:
files_fsc =  f'figures_fsc/*.h5'
if multiFSC:
    filesFSC = np.sort(np.array(glob.glob(files_fsc)))
    names_FSC = []
    combs_FSC = []
    res_FSC = []
    n_ri_inv = []

    i = 0
    for file in filesFSC:
        f_name = file.split(sep='/')[1].split(sep='.h5')[0][:-13]
        with h5py.File(file) as f:
            n_ri_inv.append(1 / f['n_ri'][:])
            combs_FSC.append(f['fsc_r'][:])
            res_FSC.append(f['fp_res_inv_nm'][:])
            if 'res_r_nm' in f:
                res_r_nm = f['res_r_nm'][0]
                write_text(f'Resolution for [{i}] {f_name}: {res_r_nm} nm\n')
            else:
                write_text(f'Resolution for [{i}] {f_name}: - nm\n')
        i += 1
        names_FSC.append(f_name)

    n_ri_inv = np.array(n_ri_inv)
    combs_FSC = np.array(combs_FSC)
    res_FSC = np.array(res_FSC)
    fp_resolution_inv = res_FSC[0,:]
    names_FSC = np.array(names_FSC)
    
    half_bit = (0.2071 + 1.9102 * n_ri_inv) / (1.2071 + 0.9102 * n_ri_inv)
    #area_under_fsc = np.trapz(combs_FSC, res_FSC, axis=1)

    #write_text(f'\n')
    #for i in range(combs_FSC.shape[0]):
    #    write_text(f'Area under FSC for {names_FSC[i]}: {area_under_fsc[i]} \n')
    
    if res_FSC.shape[0] != 0:
        plt.figure(dpi=140)

        plt.plot(fp_resolution_inv, half_bit[-1], c='k', linestyle='--', linewidth=1.0, label='_nolegend_')
        plt.axvline(x=1/0.6396483204611855, ymin=0 , ymax=1,c='k', linestyle='--', linewidth=1.0, alpha=0.4, label='_nolegend_')

        for p in range(len(filesFSC)):
            plt.plot(fp_resolution_inv, combs_FSC[p], linestyle='-')

        plt.xlim([0.0, fp_resolution_inv[-1]])
        plt.ylim([-0.15, 1.01])
        plt.xlabel('|q| $(nm^{-1})$', weight='bold')
        plt.ylabel('FSC', weight='bold')
        plt.legend(names_FSC, frameon=False, prop=dict(size=4.0, weight='bold'), loc=1);
        if saveFig:
            plt.savefig('ds_4x_fsc_all.pdf',transparent=False,bbox_inches='tight',dpi=200);

In [ ]:
# 100k patterns
filesFSC = np.sort(np.array(glob.glob(f'figures_fsc/*_100k_{extension}_*_FSC.h5')))
names_FSC = []
combs_FSC = []
res_FSC = []

i = 0
for file in filesFSC:
    f_name = file.split(sep='/')[1].split(sep='.h5')[0][:-5]
    with h5py.File(file) as f:
        combs_FSC.append(f['fsc_r'][:])
        res_FSC.append(f['fp_res_inv_nm'][:])
        if 'res_r_smooth_nm' in f:
            res_r_smooth_nm = f['res_r_smooth_nm'][()]
            write_text(f'Resolution for [{i}] {f_name}: {res_r_smooth_nm} nm\n')
        else:
            write_text(f'Resolution for [{i}] {f_name}: - nm\n')
    i += 1
    names_FSC.append(f_name)

names_FSC = np.array(names_FSC)
combs_FSC = np.array(combs_FSC)
res_FSC = np.array(res_FSC)
max_points = res_FSC.shape[1]

res_ax = res_FSC[0]

plt.figure(dpi=140)
plt.axvline(x=1/0.6396483204611855, ymin=0 , ymax=1,c='k', linestyle='--', linewidth=1.0, alpha=0.4, label='_nolegend_')

for p in range(len(filesFSC)):
    plt.plot(res_ax, combs_FSC[p], linestyle='-')

#plt.xlim([0, res_FSC[-1][-1]])
plt.ylim([None, 1.0])

plt.xlabel('|q| $(nm^{-1})$', weight='bold')
plt.ylabel('FSC', weight='bold')
plt.legend(names_FSC, frameon=True, prop=dict(size=5.0, weight='bold'), loc=1);
if saveFig:
        plt.savefig(f'fsc_{extension}_100k_all.pdf', transparent=False, bbox_inches='tight',dpi=200);

extension = 'prot_wat' # 'prot_only' or 'prot_wat'

In [ ]:
# 1M patterns
filesFSC_1M = np.sort(np.array(glob.glob(f'figures_fsc/*_1M_*_FSC.h5')))
names_FSC_1M = []
combs_FSC_1M = []
res_FSC_1M = []

i = 0
for file in filesFSC_1M:
    f_name_1M = file.split(sep='/')[1].split(sep='.h5')[0][:-5]
    with h5py.File(file) as f_1M:
        combs_FSC_1M.append(f_1M['fsc_r'][:])
        res_FSC_1M.append(f_1M['fp_res_inv_nm'][:])
        if 'res_r_smooth_nm' in f_1M:
            res_r_smooth_nm_1M= f_1M['res_r_smooth_nm'][()]
            write_text(f'Resolution for [{i}] {f_name_1M}: {res_r_smooth_nm_1M} nm\n')
        else:
            write_text(f'Resolution for [{i}] {f_name_1M}: - nm\n')
    i += 1
    names_FSC_1M.append(f_name_1M)

names_FSC_1M = np.array(names_FSC_1M)
combs_FSC_1M = np.array(combs_FSC_1M)
res_FSC_1M = np.array(res_FSC_1M)
max_points_1M = res_FSC_1M.shape[1]

res_ax_1M = res_FSC_1M[0]

plt.figure(dpi=140)
plt.axvline(x=1/0.6396483204611855, ymin=0 , ymax=1,c='k', linestyle='--', linewidth=1.0, alpha=0.4, label='_nolegend_')

for p in range(len(filesFSC_1M)):
    plt.plot(res_ax_1M, combs_FSC_1M[p], linestyle='-')

#plt.xlim([0, res_FSC_1M[-1][-1]])
plt.ylim([None, 1.0])

plt.xlabel('|q| $(nm^{-1})$', weight='bold')
plt.ylabel('PRTF', weight='bold')
plt.legend(names_FSC_1M, frameon=True, prop=dict(size=5.0, weight='bold'), loc=1);
if saveFig:
        plt.savefig(f'fsc_{extension}_1M_all.pdf', transparent=False, bbox_inches='tight',dpi=200);

In [ ]:
# 100k and 1M fill between plot for paper
fsc_min_100k = np.min(combs_FSC, axis=0)
fsc_max_100k = np.max(combs_FSC, axis=0)

fsc_min_1M = np.min(combs_FSC_1M, axis=0)
fsc_max_1M = np.max(combs_FSC_1M, axis=0)

plt.figure(dpi=140)
plt.xlim([0.0, res_FSC_1M.max(axis=1)[0]])
plt.ylim([0.0, 1.02])

plt.xlabel('|q| $(nm^{-1})$', weight='bold')
plt.ylabel('FSC', weight='bold')

plt.fill_between(res_ax, fsc_min_100k, fsc_max_100k, edgecolor="none", facecolor="b", alpha=0.8)
plt.fill_between(res_ax, fsc_min_1M, fsc_max_1M, edgecolor="none", facecolor="g", alpha=0.4)

plt.axvline(x=1/0.6396483204611855, ymin=0 , ymax=1,c='k', linestyle='--', linewidth=1.0, alpha=0.4, label='_nolegend_')

plt.legend(["100k", "1M"], prop=dict(size=9, weight="bold"), frameon=False);

if saveFig:
    plt.savefig(f'fsc_all_{extension}_filled.pdf', transparent=False, bbox_inches='tight',dpi=200);